In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
import os
import json
import torch

access_token = os.getenv('HUGGINGFACE_TOKEN')
# default: Load the model on the available device(s)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct", torch_dtype="auto", attn_implementation="flash_attention_2",device_map="auto",token=access_token
)

processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct", token=access_token)
# IMPORTANT: Fix for batch generation warnings and artifacts (like "anvas")
processor.tokenizer.padding_side = "left"

# Image List
# Directory containing the images
base_dir = "/home/dream/Study/26p1/DSproject/shenzhen/6深圳_携程图像文本/5深圳市罗湖区仙湖植物园/class_1"

# Get all image files and construct URLs
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
image_files = sorted([f for f in os.listdir(base_dir) if f.lower().endswith(valid_extensions)])
image_urls = [f"file://{os.path.join(base_dir, f)}" for f in image_files]

# Analysis Prompt - Optimized for high sensitivity (Modified to fix empty outputs)
prompt_text = """Analyze the image and extract structured data.
You represent a highly sensitive biodiversity detection system. Your goal is to detect ANY plant, animal, or insect life, and ANY human activity.
Instructions:
1. Biodiversity Detection:
   - LIST EVERY visible biological entity. 
   - Pay close attention to insects (butterflies, bees), birds, and distinct plants.
   - If exact species is unknown, use a descriptive name (e.g., "Black Butterfly", "Pink Flower", "Green Shrub"). 
   - Do NOT ignore common elements if they are the main subject.
   
2. Human Activity:
   - Identify specific actions if people are present.

Output Format: Valid JSON Only.
{
  "biodiversity": [{"species_label": "detected name", "category": "Plant/Animal/Insect"}],
  "human_activity": [{"activity_label": "activity name", "social_mode": "Solitary/Group"}]
}
"""

# START BATCH PROCESSING
BATCH_SIZE = 1  # Reduced to 1 to ensure maximum attention per image and avoid padding issues affecting quality
print(f"Total images to process: {len(image_urls)}")

for i in range(0, len(image_urls), BATCH_SIZE):
    batch_urls = image_urls[i : i + BATCH_SIZE]
    print(f"\nProcessing Batch {i//BATCH_SIZE + 1} / {(len(image_urls) + BATCH_SIZE - 1)//BATCH_SIZE} (Images {i} to {min(i+BATCH_SIZE, len(image_urls))})")
    
    # Construct batch messages
    messages_batch = []
    for url in batch_urls:
        messages_batch.append([
            {"role": "system", "content": "You are an expert biologist and ecologist. You are capable of detecting small details like insects and specific plants. Always output valid JSON."},
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": url},
                    {"type": "text", "text": prompt_text},
                ],
            }
        ])

    # Preparation for batch inference
    try:
        texts = [
            processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
            for msg in messages_batch
        ]

        image_inputs, video_inputs = process_vision_info(messages_batch)
        inputs = processor(
            text=texts,
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to("cuda")

        # Inference
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=512)
            
        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_texts = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )

        # Process and pretty print the results
        for url, output in zip(batch_urls, output_texts):
            filename = url.split("/")[-1]
            print(f"--- Result for: {filename} ---")
            try:
                # Improved cleanup logic
                clean_output = output.replace("```json", "").replace("```", "").strip()
                
                # Try to find the JSON object if there's extra noise around it
                start_idx = clean_output.find("{")
                end_idx = clean_output.rfind("}")
                if start_idx != -1 and end_idx != -1:
                    clean_output = clean_output[start_idx:end_idx+1]
                
                parsed_json = json.loads(clean_output)
                # Pretty print JSON
                print(json.dumps(parsed_json, indent=2, ensure_ascii=False))
            except json.JSONDecodeError:
                print("Raw Output (JSON parse failed):")
                print(output)
            print("-" * 30 + "\n")
            
        # Explicitly clear memory after each batch
        del inputs, generated_ids, image_inputs, video_inputs, texts, messages_batch
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"Error processing batch starting at index {i}: {str(e)}")
        # Try to clear memory even on error
        torch.cuda.empty_cache()

/home/dream/anaconda3/envs/dsproject/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 729/729 [00:00<00:00, 843.38it/s, Materializing param=model.visual.patch_embed.proj.weight]                          
Some parameters are on the meta device because they were offloaded to the cpu.
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Total images to process: 251

Processing Batch 1 / 251 (Images 0 to 1)
--- Result for: 5深圳市罗湖区仙湖植物园_233539_10.jpg ---
{
  "biodiversity": [
    {
      "species_label": "Red Azalea",
      "category": "Plant"
    },
    {
      "species_label": "Pink Azalea",
      "category": "Plant"
    },
    {
      "species_label": "Green Grass",
      "category": "Plant"
    }
  ],
  "human_activity": []
}
------------------------------


Processing Batch 2 / 251 (Images 1 to 2)
--- Result for: 5深圳市罗湖区仙湖植物园_233539_14.jpg ---
{
  "biodiversity": [
    {
      "species_label": "Butterfly Collection",
      "category": "Insect"
    }
  ],
  "human_activity": []
}
------------------------------


Processing Batch 3 / 251 (Images 2 to 3)
--- Result for: 5深圳市罗湖区仙湖植物园_233539_2.jpg ---
{
  "biodiversity": [
    {
      "species_label": "Delphinium",
      "category": "Plant"
    },
    {
      "species_label": "Green Leaves",
      "category": "Plant"
    }
  ],
  "human_activity": []
}
-----------------

KeyboardInterrupt: 

In [ ]:
# Sample messages for batch inference
messages1 = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "file:///path/to/image1.jpg"},
            {"type": "image", "image": "file:///path/to/image2.jpg"},
            {"type": "text", "text": "What are the common elements in these pictures?"},
        ],
    }
]
messages2 = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Who are you?"},
]
# Combine messages for batch processing
messages = [messages1, messages2]

# Preparation for batch inference
texts = [
    processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    for msg in messages
]
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=texts,
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")

# Batch Inference
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_texts = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_texts)
